In [2]:
import sys
!{sys.executable} -m pip install datasets

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 515 kB 1.6 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 4.2 MB/s eta 0:00:01
     |████████████████████████████████| 73 kB 3.3 MB/s eta 0:00:011
     |████████████████████████████████| 133 kB 13.0 MB/s eta 0:00:01
     |████████████████████████████████| 31.2 MB 217 kB/s eta 0:00:011
     |████████████████████████████████| 174 kB 5.8 MB/s eta 0:00:01
     |████████████████████████████████| 200 kB 9.3 MB/s eta 0:00:01
     |████████████████████████████████| 119 kB 8.4 MB/s eta 0:00:01
     |████████████████████████████████| 618 kB 4.9 MB/s eta 0:00:01
     |████████████████████████████████| 492 kB 2.6 MB/s eta 0:00:01
     |████████████████████████████████| 50 kB 5.3 MB/s eta 0:00:011
     |████████████████████████████████| 67 kB 7.8 MB/s  eta 0:00:01
     |████████████████████████████████| 44 kB 4.7 MB/s eta 0:00:01
     |███████████████████████████████

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
df = pd.DataFrame(dataset['train'])

# Konversi label angka ke nama emosi
label_map = dataset['train'].features['labels'].feature.names

def get_label(label_list):
    if len(label_list) == 0:
        return 'neutral'
    return label_map[label_list[0]]

df['emotion'] = df['labels'].apply(get_label)
df = df[['text', 'emotion']]

print(df.shape)
print(df['emotion'].value_counts())
df.to_csv('DatasetMentahGoEmotions.csv', index=False)

/Users/gabrielalevani/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/gabrielalevani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Generating test split: 100%|██████████| 5427/5427 [00:00<00:00, 1229806.46 examples/s]


(43410, 2)
emotion
neutral           12823
admiration         4130
approval           2596
amusement          2244
annoyance          2138
gratitude          2096
curiosity          1772
disapproval        1651
anger              1547
love               1533
confusion          1268
disappointment     1028
joy                1013
optimism            974
caring              966
sadness             874
surprise            751
excitement          700
realization         698
disgust             580
desire              543
fear                510
remorse             404
embarrassment       248
nervousness         105
relief               96
grief                65
pride                57
Name: count, dtype: int64


In [4]:
df = pd.read_csv('DatasetMentahGoEmotions.csv')
result = df[['text', 'emotion']]
print(result.head())
result.to_csv('MengolahData1.csv', index=False)

                                                text    emotion
0  My favourite food is anything I didn't have to...    neutral
1  Now if he does off himself, everyone will thin...    neutral
2                     WHY THE FUCK IS BAYLESS ISOING      anger
3                        To make her feel threatened       fear
4                             Dirty Southern Wankers  annoyance


In [5]:
df = pd.read_csv('MengolahData1.csv')
df['case folding'] = df['text'].str.lower()
print(df.head())
df.to_csv('MengolahData-Casefolding.csv', index=False)

                                                text    emotion  \
0  My favourite food is anything I didn't have to...    neutral   
1  Now if he does off himself, everyone will thin...    neutral   
2                     WHY THE FUCK IS BAYLESS ISOING      anger   
3                        To make her feel threatened       fear   
4                             Dirty Southern Wankers  annoyance   

                                        case folding  
0  my favourite food is anything i didn't have to...  
1  now if he does off himself, everyone will thin...  
2                     why the fuck is bayless isoing  
3                        to make her feel threatened  
4                             dirty southern wankers  


In [6]:
import string, re, nltk
nltk.download('punkt')

df = pd.read_csv('MengolahData1.csv')
MengolahData = df[['text', 'emotion']].copy()

def remove_review_special(text):
    text = re.sub(r'([@#][A-Za-z0-9]+)|(\w+:\/\/\S+)', ' ', str(text))
    return ' '.join(text.split())

def remove_number(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def remove_single_char(text):
    return re.sub(r'\b[a-zA-Z]\b', '', text)

cleaning_pipeline = [
    remove_review_special,
    remove_number,
    remove_punctuation,
    remove_whitespace,
    remove_single_char,
]

MengolahData['clean text'] = MengolahData['text']
for func in cleaning_pipeline:
    MengolahData['clean text'] = MengolahData['clean text'].apply(func)

MengolahData['case folding'] = MengolahData['text'].str.lower()
MengolahData['clean text']   = MengolahData['clean text'].str.lower()

final = MengolahData[['text', 'emotion', 'case folding', 'clean text']]
final.to_csv('Data-Clean-GoEmotions.csv', index=False)
print(f'Total baris: {len(final)}')
print(final.head())

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/gabrielalevani/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total baris: 43410
                                                text    emotion  \
0  My favourite food is anything I didn't have to...    neutral   
1  Now if he does off himself, everyone will thin...    neutral   
2                     WHY THE FUCK IS BAYLESS ISOING      anger   
3                        To make her feel threatened       fear   
4                             Dirty Southern Wankers  annoyance   

                                        case folding  \
0  my favourite food is anything i didn't have to...   
1  now if he does off himself, everyone will thin...   
2                     why the fuck is bayless isoing   
3                        to make her feel threatened   
4                             dirty southern wankers   

                                          clean text  
0  my favourite food is anything  didnt have to c...  
1  now if he does off himself everyone will think...  
2                     why the fuck is bayless isoing  
3                    

In [8]:
import sys
!{sys.executable} -m pip install scikit-learn joblib

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 11.1 MB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 3.3 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [11]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv('Data-Clean-GoEmotions.csv')

# tambah baris ini — hapus baris yang clean text-nya kosong
df = df.dropna(subset=['clean text'])
df = df[df['clean text'].str.strip() != '']

X = df['clean text']
y = df['emotion']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)} data')
print(f'Test : {len(X_test)} data')

Train: 34716 data
Test : 8680 data


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf',   LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
print("Training selesai!")

Training selesai!


In [13]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

                precision    recall  f1-score   support

    admiration       0.65      0.61      0.63       826
     amusement       0.77      0.69      0.72       449
         anger       0.59      0.31      0.41       309
     annoyance       0.30      0.08      0.13       428
      approval       0.32      0.14      0.19       519
        caring       0.36      0.17      0.23       193
     confusion       0.44      0.15      0.22       254
     curiosity       0.41      0.19      0.26       354
        desire       0.58      0.28      0.38       109
disappointment       0.36      0.04      0.08       206
   disapproval       0.37      0.11      0.17       330
       disgust       0.56      0.21      0.30       116
 embarrassment       0.67      0.04      0.08        49
    excitement       0.51      0.17      0.26       140
          fear       0.64      0.16      0.25       102
     gratitude       0.78      0.79      0.79       419
         grief       0.00      0.00      0.00  

/Users/gabrielalevani/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/gabrielalevani/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/gabrielalevani/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

In [28]:
kalimat = [
    "I am so happy today!",
    "this makes me really angry",
    "I feel so scared",
    "I don't feel anything"
]

hasil = model.predict(kalimat)
for k, h in zip(kalimat, hasil):
    print(f'{k:45} → {h}')

I am so happy today!                          → joy
this makes me really angry                    → anger
I feel so scared                              → fear
I don't feel anything                         → disapproval


In [15]:
import joblib

joblib.dump(model, 'emotion_model.pkl')
print("Model tersimpan! File: emotion_model.pkl")

Model tersimpan! File: emotion_model.pkl
